# 03. Linearity Analysis

Sixpack_Rips vs Sixpack_Chroma의 embedding 구조 차이를 검증하는 3가지 실험:
- **Exp1**: Kernel Complexity Ladder (Linear → Poly → RBF)
- **Exp2**: Classifier Hierarchy (NCM → LDA → QDA → KNN → SVM-RBF)
- **Exp4**: Linear vs RBF CKA (η = CKA_lin / CKA_rbf)

## 0. 데이터 로딩 (독립 실행용)

In [ ]:
# 01번 노트북을 먼저 실행하세요, 또는:
# %run ./01_Setup_and_Data_Loading.ipynb

## 1. Config & Common 유틸리티

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import NearestCentroid
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.preprocessing import OneHotEncoder
from scipy.spatial.distance import pdist, squareform
from scipy.stats import wilcoxon

DESCRIPTORS = ['Ord_PI', 'Inter_PI', '3D_PI', 'Sixpack_Rips', 'Sixpack_Chroma']
SEEDS = [42, 123, 456, 789, 1010]
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

COLORS = {'Ord_PI':'#4C72B0','Inter_PI':'#DD8452','3D_PI':'#55A868',
          'Sixpack_Rips':'#C44E52','Sixpack_Chroma':'#8172B3'}

def generate_cv_indices(y, seeds=SEEDS, n_splits=N_OUTER_FOLDS):
    cv_map = {}
    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for fold_idx, (tri, tei) in enumerate(skf.split(np.zeros(len(y)), y)):
            cv_map[(seed, fold_idx)] = (tri, tei)
    return cv_map

def scale_train_test(X, train_idx, test_idx, pca_dim=500):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])
    if pca_dim and X_train.shape[1] > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=42)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)
    return X_train, X_test

def compute_fisher_ratio(X, y):
    classes = np.unique(y); n, d = X.shape; grand_mean = X.mean(axis=0)
    S_W = np.zeros((d, d)); S_B = np.zeros((d, d))
    for c in classes:
        X_c = X[y == c]; n_c = len(X_c); mu_c = X_c.mean(axis=0)
        diff_c = X_c - mu_c; S_W += diff_c.T @ diff_c
        diff_m = (mu_c - grand_mean).reshape(-1, 1); S_B += n_c * (diff_m @ diff_m.T)
    eps = 1e-4 * np.mean(np.diag(S_W))
    return np.trace(np.linalg.solve(S_W + eps * np.eye(d), S_B))

print('Common utilities defined.')

## 2. Exp1: Kernel Complexity Ladder
Linear → Poly(2,3,5) → RBF(small/mid/large gamma) 순으로 kernel 복잡도를 올리며 정확도 변화 관찰.

In [ ]:
def get_kernel_ladder():
    return [
        ('Linear', SVC(kernel='linear'), {'C': [0.01, 0.1, 1, 10, 100]}),
        ('Poly-2', SVC(kernel='poly', degree=2, coef0=1), {'C': [0.1, 1, 10]}),
        ('Poly-3', SVC(kernel='poly', degree=3, coef0=1), {'C': [0.1, 1, 10]}),
        ('Poly-5', SVC(kernel='poly', degree=5, coef0=1), {'C': [0.1, 1, 10]}),
        ('RBF-small', None, {'C': [0.1, 1, 10]}),
        ('RBF-mid',   None, {'C': [0.1, 1, 10]}),
        ('RBF-large', None, {'C': [0.1, 1, 10]}),
    ]

def run_exp1(datasets, descriptor_list):
    records = []
    for desc in descriptor_list:
        if desc not in datasets: print(f'  [SKIP] {desc}'); continue
        X, y = datasets[desc]['X'], datasets[desc]['y']
        d = X.shape[1]; cv_map = generate_cv_indices(y)
        print(f'\n[Exp1] {desc} (dim={d})')
        ladder = get_kernel_ladder()
        for seed in SEEDS:
            for fold_idx in range(N_OUTER_FOLDS):
                tri, tei = cv_map[(seed, fold_idx)]
                X_train, X_test = scale_train_test(X, tri, tei)
                y_train, y_test = y[tri], y[tei]
                for kern_name, base_clf, param_grid in ladder:
                    if kern_name == 'RBF-small': base_clf = SVC(kernel='rbf', gamma=0.1/d)
                    elif kern_name == 'RBF-mid': base_clf = SVC(kernel='rbf', gamma=1.0/d)
                    elif kern_name == 'RBF-large': base_clf = SVC(kernel='rbf', gamma=10.0/d)
                    gs = GridSearchCV(base_clf, param_grid, cv=N_INNER_FOLDS,
                                      scoring='accuracy', n_jobs=-1, refit=True)
                    gs.fit(X_train, y_train)
                    acc = accuracy_score(y_test, gs.predict(X_test))
                    records.append({'descriptor':desc,'kernel':kern_name,'seed':seed,
                                    'fold':fold_idx,'accuracy':acc*100,'best_C':gs.best_params_.get('C')})
        df_tmp = pd.DataFrame([r for r in records if r['descriptor']==desc])
        for kern, row in df_tmp.groupby('kernel')['accuracy'].agg(['mean','std']).iterrows():
            print(f"  {kern:<12s} {row['mean']:.2f} ± {row['std']:.2f}%")
    return pd.DataFrame(records)

df_exp1 = run_exp1(datasets, [d for d in DESCRIPTORS if d in datasets])

## 3. Exp2: Classifier Hierarchy
NCM → LDA → QDA → KNN → SVM-RBF 순으로 r_LDA = Acc_LDA / Acc_SVM-RBF 계산.

In [ ]:
def get_classifier_hierarchy():
    return [
        ('NCM', NearestCentroid(), {}),
        ('LDA', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'), {}),
        ('QDA', QuadraticDiscriminantAnalysis(reg_param=0.01), {}),
        ('KNN', KNeighborsClassifier(), {'n_neighbors': [1, 5, 10]}),
        ('SVM-RBF', SVC(kernel='rbf', gamma='scale'), {'C': [0.1, 1, 10]}),
    ]

def run_exp2(datasets, descriptor_list):
    records = []
    for desc in descriptor_list:
        if desc not in datasets: print(f'  [SKIP] {desc}'); continue
        X, y = datasets[desc]['X'], datasets[desc]['y']
        cv_map = generate_cv_indices(y)
        print(f'\n[Exp2] {desc} (dim={X.shape[1]})')
        for seed in SEEDS:
            for fold_idx in range(N_OUTER_FOLDS):
                tri, tei = cv_map[(seed, fold_idx)]
                X_train, X_test = scale_train_test(X, tri, tei)
                y_train, y_test = y[tri], y[tei]
                J = compute_fisher_ratio(X_train, y_train)
                for clf_name, base_clf, param_grid in get_classifier_hierarchy():
                    if param_grid:
                        gs = GridSearchCV(base_clf, param_grid, cv=N_INNER_FOLDS,
                                          scoring='accuracy', n_jobs=-1, refit=True)
                        gs.fit(X_train, y_train)
                        acc = accuracy_score(y_test, gs.predict(X_test))
                    else:
                        clf = clone(base_clf); clf.fit(X_train, y_train)
                        acc = accuracy_score(y_test, clf.predict(X_test))
                    records.append({'descriptor':desc,'classifier':clf_name,'seed':seed,
                                    'fold':fold_idx,'accuracy':acc*100,'fisher_J':J})
        df_tmp = pd.DataFrame([r for r in records if r['descriptor']==desc])
        pivot = df_tmp.groupby('classifier')['accuracy'].mean()
        rbf_acc = pivot.get('SVM-RBF', 1)
        print(f"  NCM={pivot.get('NCM',0):.2f}  LDA={pivot.get('LDA',0):.2f}  "
              f"QDA={pivot.get('QDA',0):.2f}  KNN={pivot.get('KNN',0):.2f}  RBF={rbf_acc:.2f}")
        print(f"  r_LDA={pivot.get('LDA',0)/rbf_acc:.4f}  J={df_tmp['fisher_J'].mean():.4f}")
    return pd.DataFrame(records)

df_exp2 = run_exp2(datasets, [d for d in DESCRIPTORS if d in datasets])

## 4. Exp4: Linear vs RBF CKA
η = CKA_linear / CKA_RBF → 1에 가까울수록 linear 구조.

In [ ]:
def linear_cka(X, Y_onehot):
    Xc = X - X.mean(axis=0); Yc = Y_onehot - Y_onehot.mean(axis=0)
    num = np.linalg.norm(Yc.T @ Xc, 'fro') ** 2
    denom = np.linalg.norm(Xc.T @ Xc, 'fro') * np.linalg.norm(Yc.T @ Yc, 'fro')
    return num / (denom + 1e-10)

def rbf_cka(X, Y_onehot, subsample=4000, rng=None):
    n = X.shape[0]
    if n > subsample and rng is not None:
        idx = rng.choice(n, subsample, replace=False); X, Y_onehot = X[idx], Y_onehot[idx]; n = subsample
    D2 = squareform(pdist(X, 'sqeuclidean'))
    sigma2 = np.median(D2[np.triu_indices(n, k=1)])
    if sigma2 < 1e-10: sigma2 = 1.0
    K_X = np.exp(-D2 / (2 * sigma2)); K_Y = Y_onehot @ Y_onehot.T
    H = np.eye(n) - np.ones((n, n)) / n
    HKxH = H @ K_X @ H; HKyH = H @ K_Y @ H
    num = np.sum(HKxH * HKyH); denom = np.sqrt(np.sum(HKxH**2) * np.sum(HKyH**2))
    return num / (denom + 1e-10)

def run_exp4(datasets, descriptor_list):
    records = []; enc = OneHotEncoder(sparse_output=False)
    for desc in descriptor_list:
        if desc not in datasets: print(f'  [SKIP] {desc}'); continue
        X, y = datasets[desc]['X'], datasets[desc]['y']
        cv_map = generate_cv_indices(y)
        Y_oh_full = enc.fit_transform(y.reshape(-1, 1))
        print(f'\n[Exp4] {desc} (dim={X.shape[1]})')
        for seed in SEEDS:
            rng = np.random.RandomState(seed)
            for fold_idx in range(N_OUTER_FOLDS):
                tri, tei = cv_map[(seed, fold_idx)]
                X_train, _ = scale_train_test(X, tri, tei); Y_train = Y_oh_full[tri]
                cka_lin = linear_cka(X_train, Y_train)
                cka_rbf_val = rbf_cka(X_train, Y_train, rng=rng)
                records.append({'descriptor':desc,'seed':seed,'fold':fold_idx,
                                'cka_linear':cka_lin,'cka_rbf':cka_rbf_val,
                                'eta':cka_lin/(cka_rbf_val+1e-10)})
        df_tmp = pd.DataFrame([r for r in records if r['descriptor']==desc])
        print(f"  CKA_lin={df_tmp['cka_linear'].mean():.4f}  "
              f"CKA_rbf={df_tmp['cka_rbf'].mean():.4f}  η={df_tmp['eta'].mean():.4f}")
    return pd.DataFrame(records)

df_exp4 = run_exp4(datasets, [d for d in DESCRIPTORS if d in datasets])

## 5. Reporting — Figure 생성

In [ ]:
# Figure 1: Kernel Ladder
kernel_order = ['Linear','Poly-2','Poly-3','Poly-5','RBF-small','RBF-mid','RBF-large']
fig, ax = plt.subplots(figsize=(12, 6))
for desc in DESCRIPTORS:
    sub = df_exp1[df_exp1['descriptor']==desc]
    if sub.empty: continue
    means = [sub[sub['kernel']==k]['accuracy'].mean() for k in kernel_order]
    stds  = [sub[sub['kernel']==k]['accuracy'].std() for k in kernel_order]
    ax.errorbar(range(len(kernel_order)), means, yerr=stds, fmt='o-', label=desc,
               color=COLORS.get(desc,'gray'), linewidth=2, markersize=7, capsize=3)
ax.set_xticks(range(len(kernel_order))); ax.set_xticklabels(kernel_order, rotation=30, ha='right')
ax.set_xlabel('Kernel (complexity →)'); ax.set_ylabel('Strict Accuracy (%)')
ax.set_title('Exp 1: Kernel Complexity Ladder', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_kernel_ladder.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Classifier Hierarchy
clf_order = ['NCM','LDA','QDA','KNN','SVM-RBF']
fig, ax = plt.subplots(figsize=(14, 6)); x = np.arange(len(clf_order)); width = 0.15
for i, desc in enumerate(DESCRIPTORS):
    sub = df_exp2[df_exp2['descriptor']==desc]
    if sub.empty: continue
    means = [sub[sub['classifier']==c]['accuracy'].mean() for c in clf_order]
    stds  = [sub[sub['classifier']==c]['accuracy'].std() for c in clf_order]
    ax.bar(x+i*width, means, width, yerr=stds, label=desc,
           color=COLORS.get(desc,'gray'), capsize=2)
ax.set_xticks(x + width*2); ax.set_xticklabels(clf_order)
ax.set_ylabel('Strict Accuracy (%)'); ax.set_title('Exp 2: Classifier Hierarchy', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_classifier_hierarchy.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: CKA
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, title in zip(axes, ['cka_linear','cka_rbf','eta'],
                              ['CKA Linear','CKA RBF','η = CKA_lin / CKA_rbf']):
    means, stds, names = [], [], []
    for desc in DESCRIPTORS:
        sub = df_exp4[df_exp4['descriptor']==desc]
        if sub.empty: continue
        names.append(desc); means.append(sub[metric].mean()); stds.append(sub[metric].std())
    ax.bar(range(len(names)), means, yerr=stds,
           color=[COLORS.get(n,'gray') for n in names], capsize=3)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold'); ax.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(means): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=8)
plt.suptitle('Exp 4: CKA Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_cka.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Paired Wilcoxon Test (Chroma vs Rips)

In [ ]:
print('--- Paired Wilcoxon (Chroma vs Rips) ---')
lin_c = df_exp1[(df_exp1['descriptor']=='Sixpack_Chroma') & (df_exp1['kernel']=='Linear')]
lin_r = df_exp1[(df_exp1['descriptor']=='Sixpack_Rips') & (df_exp1['kernel']=='Linear')]
if len(lin_c)==len(lin_r) and len(lin_c)>=5:
    s, p = wilcoxon(lin_c['accuracy'].values, lin_r['accuracy'].values)
    print(f'  Exp1 Linear kernel: W={s}, p={p:.4f}')

eta_c = df_exp4[df_exp4['descriptor']=='Sixpack_Chroma'].sort_values(['seed','fold'])['eta'].values
eta_r = df_exp4[df_exp4['descriptor']=='Sixpack_Rips'].sort_values(['seed','fold'])['eta'].values
if len(eta_c)==len(eta_r) and len(eta_c)>=5:
    s, p = wilcoxon(eta_c, eta_r)
    print(f'  Exp4 η: W={s}, p={p:.4f}')